#Chuyển file json sang cvs để lọc 5 core

In [1]:
import os
import pandas as pd

In [2]:
PATH = "/content/drive/MyDrive/Colab Notebooks/CTH001/AmazonData"

In [3]:
!cp "{PATH}/2023/df_review_2023.in_meta.jsonl" "df_review_2023.in_meta.jsonl"

In [ ]:
# Đường dẫn file của bạn
file_path = "df_review_2023.in_meta.jsonl"

# Cách sửa: Mở file trước rồi mới đọc
with open(file_path, 'r', encoding='utf-8') as f:
    review_chunks = pd.read_json(f, lines=True, chunksize=100_000)
    df = next(review_chunks)

# Hiển thị DataFrame
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   rating             100000 non-null  int64         
 1   title              100000 non-null  object        
 2   text               100000 non-null  object        
 3   images             100000 non-null  object        
 4   asin               100000 non-null  object        
 5   parent_asin        100000 non-null  object        
 6   user_id            100000 non-null  object        
 7   timestamp          100000 non-null  datetime64[ns]
 8   helpful_vote       100000 non-null  int64         
 9   verified_purchase  100000 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(2), object(6)
memory usage: 7.0+ MB


In [ ]:
df.head(3)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,Good buy for preschool naps and home use...,I bought two of these for my kids for nap time...,[],B004FM7VOW,B089MS68G8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 18:52:17,1,True
1,5,THEY WORK- and are super cute to boot...,LOVE THESE! AND THEY WORK!!! I was on the fenc...,[],B01E5E703G,B01E5E703G,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 17:44:04,1,True
2,1,cute but small and pretty much unusable as a c...,cute but small and pretty much unusable as a c...,[],B00F463XV8,B00F9386Q8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-01-13 02:08:01,0,True


In [ ]:
df_rating = df[["user_id", "parent_asin", "rating", "timestamp"]]

In [ ]:
df_rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   user_id      100000 non-null  object        
 1   parent_asin  100000 non-null  object        
 2   rating       100000 non-null  int64         
 3   timestamp    100000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 3.1+ MB


In [ ]:
# df_rating.to_parquet('df_rating.parquet')

In [ ]:
import pandas as pd
import os

# 1. Cấu hình đường dẫn
input_path = "df_review_2023.in_meta.jsonl"
output_path = "df_rating_2023.csv" # Hoặc .jsonl tùy bạn chọn

# Xóa file cũ nếu đã tồn tại để tránh ghi đè lặp dữ liệu
if os.path.exists(output_path):
    os.remove(output_path)

# 2. Định nghĩa các cột cần thiết và tên mới
columns_to_keep = ["user_id", "parent_asin", "rating", "timestamp"]
rename_map = {
    "user_id": "userID",
    "parent_asin": "itemID",
    "rating": "rating",
    "timestamp": "timestamp"
}

print("🚀 Đang bắt đầu xử lý từng cụm 100,000 dòng...")

# 3. Mở file và quét theo chunk
with open(input_path, 'r', encoding='utf-8') as f:
    review_chunks = pd.read_json(f, lines=True, chunksize=100_000)

    for i, chunk in enumerate(review_chunks):
        # Lọc cột và đổi tên
        chunk_clean = chunk[columns_to_keep].rename(columns=rename_map)

        # Ghi xuống file (Dùng CSV để nhẹ và dễ kiểm tra, hoặc JSONL)
        # mode='a' là append (nối đuôi), header=True chỉ cho chunk đầu tiên
        chunk_clean.to_csv(output_path, mode='a', index=False, header=(i == 0))

        if (i + 1) % 5 == 0:
            print(f"✅ Đã xử lý xong {(i + 1) * 100_000} dòng...")

print(f"✨ Tất cả dữ liệu đã được lọc và ghi ra: {output_path}")

🚀 Đang bắt đầu xử lý từng cụm 100,000 dòng...
✅ Đã xử lý xong 500000 dòng...
✅ Đã xử lý xong 1000000 dòng...
✅ Đã xử lý xong 1500000 dòng...
✅ Đã xử lý xong 2000000 dòng...
✅ Đã xử lý xong 2500000 dòng...
✅ Đã xử lý xong 3000000 dòng...
✅ Đã xử lý xong 3500000 dòng...
✅ Đã xử lý xong 4000000 dòng...
✅ Đã xử lý xong 4500000 dòng...
✅ Đã xử lý xong 5000000 dòng...
✅ Đã xử lý xong 5500000 dòng...
✅ Đã xử lý xong 6000000 dòng...
✨ Tất cả dữ liệu đã được lọc và ghi ra: interactions_2023.csv


In [ ]:
!cp "df_rating_2023.csv" "{PATH}/2023/df_rating_2023.csv"

In [4]:
import polars as pl

input_path = "df_review_2023.in_meta.jsonl"
output_path = "df_rating_2023.parquet"

print("🚀 Đang bắt đầu chuyển đổi sang Parquet bằng cơ chế Streaming...")

# 1. Sử dụng LazyFrame để quét file (không nạp vào RAM ngay)
(
    pl.scan_ndjson(input_path)
    # 2. Chọn cột và đổi tên ngay từ bước đọc (tương đương usecols)
    .select([
        pl.col("user_id").alias("userID"),
        pl.col("parent_asin").alias("itemID"),
        pl.col("rating"),
        pl.col("timestamp")
    ])
    # 3. Ghi trực tiếp ra Parquet theo kiểu luồng (Streaming)
    # sink_parquet giúp ghi dữ liệu xuống đĩa mà không cần nạp toàn bộ vào RAM
    .sink_parquet(output_path, compression="snappy")
)

print(f"✅ Đã ghi file Parquet thành công: {output_path}")

# 4. Kiểm tra nhanh kết quả
final_df = pl.scan_parquet(output_path)
print(f"📊 Tổng số tương tác: {final_df.select(pl.len()).collect().item()}")
print(f"📋 Cấu trúc cột: {final_df.collect_schema().names()}")

🚀 Đang bắt đầu chuyển đổi sang Parquet bằng cơ chế Streaming...
✅ Đã ghi file Parquet thành công: df_rating_2023.parquet
📊 Tổng số tương tác: 6025848
📋 Cấu trúc cột: ['userID', 'itemID', 'rating', 'timestamp']


In [5]:
df_fin = final_df.collect().to_pandas()

In [6]:
df_fin.shape

(6025848, 4)

In [7]:
df_fin.to_parquet('df_rating_2023.parquet')

In [9]:
!cp "df_rating_2023.parquet" "{PATH}/2023/df_rating_2023.parquet"

In [8]:
!ls -lh

total 2.9G
-rw-r--r-- 1 root root 173M Mar 30 02:53 df_rating_2023.parquet
-rw------- 1 root root 2.7G Mar 30 02:53 df_review_2023.in_meta.jsonl
drwx------ 5 root root 4.0K Mar 30 02:51 drive
drwxr-xr-x 1 root root 4.0K Mar 23 13:29 sample_data
